In [ ]:

import pandas as pd
import requests
import io
from google.colab import files

# 1. Upload dataset
print("Step 1: Upload 'CrimeData_Final_Expanded_500.csv'")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# 2. Prepare Dates for API
print("Processing dates...")
try:
    df['date_obj'] = pd.to_datetime(df['date'], format='%d.%m.%Y')
except:
    df['date_obj'] = pd.to_datetime(df['date'])

min_date = df['date_obj'].min().strftime('%Y-%m-%d')
max_date = df['date_obj'].max().strftime('%Y-%m-%d')

print(f"Fetching real weather for Kandy from {min_date} to {max_date}...")

# 3. Call Open-Meteo Archive API
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 7.2906,
    "longitude": 80.6337,
    "start_date": min_date,
    "end_date": max_date,
    "daily": ["weather_code", "precipitation_sum"],
    "timezone": "auto"
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()

    weather_df = pd.DataFrame({
        'date_obj': pd.to_datetime(data['daily']['time']),
        'wmo_code': data['daily']['weather_code'],
        'precip': data['daily']['precipitation_sum']
    })

    # 4. Map WMO Codes to Text (Sunny/Cloudy/Rainy)
    def get_weather_desc(row):
        code = row['wmo_code']
        if code == 0: return "Sunny"
        if code in [1, 2, 3]: return "Cloudy"
        if code in [45, 48]: return "Cloudy"
        if code >= 51: return "Rainy"
        return "Cloudy"

    weather_df['weather'] = weather_df.apply(get_weather_desc, axis=1)

    merged_df = pd.merge(df, weather_df[['date_obj', 'weather']], on='date_obj', how='left')

    if 'date_obj' in merged_df.columns: del merged_df['date_obj']


    output_filename = "CrimeData_Real_Weather_Kandy.csv"
    merged_df.to_csv(output_filename, index=False)

    print(f"\nSuccess! Merged {len(weather_df)} days of real weather data.")
    print("Downloading final file...")
    files.download(output_filename)

else:
    print("Error fetching weather data.")
    print(response.text)

Step 1: Upload 'CrimeData_Final_Expanded_500.csv'


Saving CrimeData.csv to CrimeData.csv
Processing dates...
Fetching real weather for Kandy from 2019-12-31 to 2025-09-23...

Success! Merged 2094 days of real weather data.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>